# Model 01 — Decision Tree
Baseline model using Decision Tree Regressor

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.tree import DecisionTreeRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

df_train = pd.concat([df24, df25], ignore_index=True)
df_test  = df26.copy()

TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]

X_train = df_train[num_cols].values
y_train = df_train[TARGET].astype(float).values
X_test  = df_test[num_cols].values
y_test  = df_test[TARGET].astype(float).values

imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(X_train)
X_test  = imp.transform(X_test)



In [ ]:
# Train Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("results")
print(f"MAE  : {mae:.4f} days")
print(f"RMSE : {rmse:.4f} days")
print(f"R2   : {r2:.4f}")
print()
print("high R2 which means there could be a problem.")


In [ ]:
# Investigate feature importance
import matplotlib.pyplot as plt
importances = dt.feature_importances_
feat_imp = pd.Series(importances, index=num_cols).sort_values(ascending=False)
print("features")
print(feat_imp.head(10))

fig, ax = plt.subplots(figsize=(10,6))
feat_imp.head(15).plot(kind='bar', color='steelblue', edgecolor='k', ax=ax)
ax.set_title('Decision Tree — Feature Importance')
ax.set_ylabel('Importance Score')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('model_01_feature_importance.png', dpi=150)
plt.show()


In [ ]:

if 'Annual_Rounds' in num_cols:
    ar_imp = feat_imp.get('Annual_Rounds', 0)
    mi_imp = feat_imp.get('Months_In_Season', 0)
    print(f"Annual_Rounds importance    : {ar_imp:.4f}")
    print(f"Months_In_Season importance : {mi_imp:.4f}")
    print()
    print("leakage- Annual_Rounds and Months_In_Season are dominating.")
    print("These columns mathematically encode Target_Days.")
    print("Action: Remove them from all preprocessing — see pre_02_drop_leakage.ipynb")


##  Leakage there
- Decision Tree shows suspiciously perfect performance because leakage columns are present
- `Annual_Rounds` and `Months_In_Season` are the top features — they encode the answer directly
